# NBA Player Longevity Prediction — Feature Engineering

**Author:** Lapele Kenneth  
**Program:** 3MTT AI & Machine Learning Fellowship  
**Dataset:** NBA Players Stats (1340 players, 22 columns)

---

## Project Goal

Predict whether an NBA player will last **5 or more years** in the league (`target_5yrs = 1`) based on their early-career performance statistics. This notebook covers:

1. Loading the dataset and defining the target variable
2. Dropping non-predictive / noise columns
3. Correlation analysis to reduce multicollinearity
4. Engineering new composite features
5. Handling missing values
6. Producing a clean, model-ready dataset

## Step 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
print('Libraries loaded successfully ✓')

## Step 2 — Load Dataset & Define Target Variable

The dataset contains statistics for **1,340 NBA players**. The column `target_5yrs` is our **dependent variable (target)**:
- `1` → the player remained in the NBA for 5+ years  
- `0` → the player did not reach 5 years

All other feature columns are potential predictors (independent variables).

In [ ]:
df = pd.read_csv('nba_players.csv')

print('Dataset shape:', df.shape)
print('\nColumn names:', df.columns.tolist())
df.head()

In [ ]:
# Isolate the target variable
target = df['target_5yrs']

print('Target variable distribution:')
print(target.value_counts())
print(f'\nClass balance — 5yr+ players: {target.mean()*100:.1f}%')

## Step 3 — Drop Non-Predictive Columns

**Columns removed and rationale:**

| Column | Reason for Dropping |
|--------|---------------------|
| `Unnamed: 0` | Row index — carries no statistical information |
| `name` | Player name — would cause **data leakage** if a model memorised specific players; also useless for generalisation |

These columns add noise and could introduce leakage without contributing any predictive signal.

In [ ]:
df_clean = df.drop(columns=['Unnamed: 0', 'name'])

print('Columns after dropping noise columns:')
print(df_clean.columns.tolist())
print('\nNew shape:', df_clean.shape)

## Step 4 — Correlation Analysis & Multicollinearity Reduction

Many basketball statistics are naturally correlated — for example, a player who scores more points will likely also attempt more field goals. High multicollinearity between features can destabilise machine learning models (especially linear models) and inflate coefficients.

We compute a correlation matrix and flag any feature pairs with |r| > 0.85 as candidates for removal.

In [ ]:
# Compute correlation matrix on features only (exclude target)
features_only = df_clean.drop(columns=['target_5yrs'])
corr_matrix = features_only.corr()

# Plot heatmap
plt.figure(figsize=(14, 11))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, linewidths=0.5,
    annot_kws={'size': 8}
)
plt.title('Feature Correlation Heatmap (lower triangle)', fontsize=14)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Heatmap saved.')

In [ ]:
# Identify highly correlated pairs
THRESHOLD = 0.85
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

high_corr_pairs = [
    (col, row, upper_tri.loc[row, col])
    for col in upper_tri.columns
    for row in upper_tri.index
    if pd.notna(upper_tri.loc[row, col]) and abs(upper_tri.loc[row, col]) > THRESHOLD
]

print(f'Feature pairs with |r| > {THRESHOLD}:')
for f1, f2, r in sorted(high_corr_pairs, key=lambda x: abs(x[2]), reverse=True):
    print(f'  {f1:10s} ↔ {f2:10s}  r = {r:.3f}')

### Redundant Columns Identified

Based on the correlation analysis, the following columns are **highly redundant** and will be dropped:

| Dropped Column | Kept Column | Reason |
|----------------|-------------|--------|
| `fgm` | `fg` (field goal %) | `fgm` ↔ `pts` r=0.99; efficiency % is more informative than raw makes |
| `fga` | `fg` (field goal %) | `fga` ↔ `pts` r=0.98; attempts redundant when % is kept |
| `ftm` | `ft` (free throw %) | `ftm` ↔ `fta` r=0.98; % captures shooting quality |
| `fta` | `ft` (free throw %) | Same rationale as above |
| `3p_made` | `3p` (3-point %) | `3p_made` ↔ `3pa` r=0.98; % preferred |
| `3pa` | `3p` (3-point %) | Same rationale as above |
| `dreb` | `reb` | `dreb` ↔ `reb` r=0.98; total rebounds subsumes defensive |
| `oreb` | `reb` | `oreb` ↔ `reb` r=0.93; total rebounds subsumes offensive |

> **Strategy:** We keep the **percentage/efficiency** versions of shooting stats (fg%, ft%, 3p%) over raw counts, as they normalise for playing time and better reflect skill. We keep total rebounds (`reb`) over its components.

In [ ]:
redundant_cols = ['fgm', 'fga', 'ftm', 'fta', '3p_made', '3pa', 'dreb', 'oreb']

df_reduced = df_clean.drop(columns=redundant_cols)

print('Remaining columns after removing redundant features:')
print(df_reduced.columns.tolist())
print('\nShape:', df_reduced.shape)

## Step 5 — Engineer Composite Features

Raw statistics don't always capture player *impact* efficiently. We engineer new features that combine multiple stats into single, meaningful metrics.

### Feature 1: Points Per Minute (`pts_per_min`)
**Formula:** `pts / min`  
**Rationale:** Normalises scoring by playing time — a player scoring 15 pts in 20 minutes is more efficient than one scoring 15 pts in 35 minutes. This directly captures scoring efficiency regardless of role or team system.

### Feature 2: Player Efficiency Rating — Simplified (`efficiency_rating`)
**Formula:** `(pts + reb + ast + stl + blk) - tov`  
**Rationale:** Summarises overall positive contributions (points, rebounds, assists, steals, blocks) minus turnovers. This is a simplified version of the NBA's official PER — widely used as a composite measure of all-round performance.

### Feature 3: Assist-to-Turnover Ratio (`ast_tov_ratio`)
**Formula:** `ast / (tov + 0.001)` *(small epsilon to avoid division by zero)*  
**Rationale:** A high ast/tov ratio signals basketball IQ and ball-handling quality — two traits strongly associated with longevity, particularly for guards and playmakers.

In [ ]:
df_engineered = df_reduced.copy()

# Feature 1: Points Per Minute
df_engineered['pts_per_min'] = df_engineered['pts'] / df_engineered['min'].replace(0, np.nan)

# Feature 2: Simplified Player Efficiency Rating
df_engineered['efficiency_rating'] = (
    df_engineered['pts'] +
    df_engineered['reb'] +
    df_engineered['ast'] +
    df_engineered['stl'] +
    df_engineered['blk'] -
    df_engineered['tov']
)

# Feature 3: Assist-to-Turnover Ratio
df_engineered['ast_tov_ratio'] = df_engineered['ast'] / (df_engineered['tov'] + 0.001)

print('New engineered features added:')
print(df_engineered[['pts_per_min', 'efficiency_rating', 'ast_tov_ratio']].describe().round(3))

In [ ]:
# Visualise distribution of engineered features by target
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
new_features = ['pts_per_min', 'efficiency_rating', 'ast_tov_ratio']
titles = ['Points Per Minute', 'Efficiency Rating', 'Assist/Turnover Ratio']

for ax, feat, title in zip(axes, new_features, titles):
    for label, color in [(0, '#e74c3c'), (1, '#2ecc71')]:
        subset = df_engineered[df_engineered['target_5yrs'] == label][feat].dropna()
        ax.hist(subset, bins=30, alpha=0.6, color=color,
                label='< 5 years' if label == 0 else '5+ years')
    ax.set_title(title, fontsize=12)
    ax.set_xlabel(feat)
    ax.set_ylabel('Count')
    ax.legend()

plt.suptitle('Engineered Feature Distributions by Target Class', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('engineered_features_dist.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 6 — Handle Missing Values

Before we can pass the dataset to a machine learning model, we must ensure there are no null values that could cause errors or introduce bias.

**Strategy:**
- Check for nulls across all columns
- For any nulls introduced during feature engineering (e.g., `pts_per_min` when `min = 0`), impute with the **median** — robust to outliers and does not distort the distribution as mean imputation might
- We do **not** use forward-fill or drop rows, as that would lose training data

In [ ]:
print('Null value counts per column:')
null_counts = df_engineered.isnull().sum()
print(null_counts[null_counts > 0] if null_counts.sum() > 0 else 'No null values found in original columns.')

# Impute any nulls in engineered features with column median
for col in ['pts_per_min', 'efficiency_rating', 'ast_tov_ratio']:
    n_nulls = df_engineered[col].isnull().sum()
    if n_nulls > 0:
        median_val = df_engineered[col].median()
        df_engineered[col].fillna(median_val, inplace=True)
        print(f'Imputed {n_nulls} null(s) in `{col}` with median = {median_val:.4f}')

print('\nFinal null counts after imputation:')
print(df_engineered.isnull().sum().sum(), 'total nulls remaining')

## Step 7 — Final Dataset Overview

We now have a clean, reduced, feature-engineered dataset ready for machine learning.

In [ ]:
print('='*55)
print('FINAL DATASET SUMMARY')
print('='*55)
print(f'Rows         : {df_engineered.shape[0]}')
print(f'Features     : {df_engineered.shape[1] - 1}  (excluding target)')
print(f'Target column: target_5yrs')
print(f'Null values  : {df_engineered.isnull().sum().sum()}')
print()
print('Feature columns:')
for i, col in enumerate([c for c in df_engineered.columns if c != 'target_5yrs'], 1):
    print(f'  {i:2d}. {col}')
print()
df_engineered.head()

In [ ]:
# Separate X and y
X = df_engineered.drop(columns=['target_5yrs'])
y = df_engineered['target_5yrs']

print('X (features) shape:', X.shape)
print('y (target) shape  :', y.shape)
print('\nTarget class distribution:')
print(y.value_counts().rename({0: 'Under 5yrs', 1: '5yrs+'}))

# Save clean dataset
df_engineered.to_csv('nba_clean_features.csv', index=False)
print('\nClean dataset saved to: nba_clean_features.csv ✓')

## Summary of Feature Engineering Decisions

| Decision | Action | Justification |
|----------|--------|---------------|
| Target isolation | Set `target_5yrs` as `y` | Binary classification target |
| Remove index | Dropped `Unnamed: 0` | Pure row index, no signal |
| Remove identifier | Dropped `name` | Would cause data leakage; no predictive value |
| Multicollinearity | Dropped `fgm, fga, ftm, fta, 3p_made, 3pa, dreb, oreb` | All had r > 0.85 with retained features; shooting % metrics preferred |
| New feature | `pts_per_min` | Normalises scoring efficiency by minutes played |
| New feature | `efficiency_rating` | Composite impact metric combining positive and negative contributions |
| New feature | `ast_tov_ratio` | Captures basketball IQ and ball control quality |
| Missing values | Median imputation | Robust to outliers; no row deletion to preserve training data |

---

The resulting dataset has **14 features** (11 original + 3 engineered), is free of null values, and is formatted for direct use with scikit-learn or any ML framework.